# **1. Data Collection**

API posts:
1.   /api/posts/search
2.   main parameters: subreddit, query, title, after, before, limit, fields.

API comments:
1.   /api/comments/search
2.   main parameters: link_id, limit, fields.

JSON fields:
1.   posts: id, title, selftext, created_utc, subreddit
2.   comments: id, link_id, parent_id, body, created_utc

parameter reference:
* https://github.com/ArthurHeitmann/arctic_shift









  


## **1.1 data collection (posts)**

In [ ]:
import requests
import time
import json
import os

headers = {
    "User-Agent": "Research_Project"
}

post_url = "https://arctic-shift.photon-reddit.com/api/posts/search"

# 10-year period:
start = int(time.mktime(time.strptime("2016-01-01 00:00:00", "%Y-%m-%d %H:%M:%S")))
end = int(time.mktime(time.strptime("2025-12-31 00:00:00", "%Y-%m-%d %H:%M:%S")))

# 30-day:
step = 60 * 60 * 24 * 30

if not os.path.exists("raw data"):
    os.makedirs("raw data")

all_threads = []
seen_ids = set()

def fetch_range(after_t, before_t):

    current_before = before_t
    batch_num = 1

    while current_before > after_t:

        params = {
            "subreddit": "Aphantasia",
            "before": current_before,
            "after": after_t,
            "sort": "desc",
            "limit": 100,
            "fields": "id,title,selftext,created_utc,subreddit,score,num_comments"
        }

        response = requests.get(post_url, params=params, headers=headers)

        if response.status_code != 200:
            print("Failed:", response.status_code)
            break

        posts = response.json().get("data", [])

        if not posts:
            break

        for post in posts:
            post_id = post.get("id")

            if post_id in seen_ids:
                continue

            seen_ids.add(post_id)

            all_threads.append({"post": post})

        last_ts = posts[-1]["created_utc"]

        print(f"Batch {batch_num} | collected: {len(all_threads)}")

        current_before = last_ts - 1
        batch_num += 1

        time.sleep(0.5)



cursor = start

while cursor < end:

    next_cursor = min(cursor + step, end)

    print("\n=== Range ===")
    print(cursor, "→", next_cursor)

    fetch_range(cursor, next_cursor)

    cursor = next_cursor



output_file = "original_posts.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_threads, f, ensure_ascii=False, indent=4)


print("Total posts:", len(all_threads))

# **2. Data filtering**



## 2.0 word frequency

to filter high-frequency keywords related with reading

In [ ]:
!pip install pandas
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import json
from collections import Counter
import pandas as pd
import spacy


nlp = spacy.load("en_core_web_sm")


with open("original_posts.json", "r", encoding="utf-8") as f:
    data = json.load(f)

word_counter = Counter()

for item in data:

    title = item["post"].get("title", "")
    selftext = item["post"].get("selftext", "")

    # Combine title and content
    text = title + " " + selftext

    if not text.strip():
        continue

    doc = nlp(text)

    for token in doc:

        if token.pos_ not in ["NOUN", "PROPN"]:
            continue

        if token.is_stop:
            continue

        if token.is_punct or token.like_num or token.is_space:
            continue

        word = token.lemma_.lower().strip()

        if len(word) < 3:
            continue

        word_counter[word] += 1


# Top 350 words:
top_words = word_counter.most_common(350)

df = pd.DataFrame(top_words, columns=["Word", "Frequency"])

df.insert(0, "Rank", range(1, len(df) + 1))

print(df)


df.to_csv(
    "high_frequency_words.csv",
    index=False,
    encoding="utf-8-sig"
)


print("Results saved to high_frequency_words.csv")

## **2.1 filtered posts**

first filtering: all keywords


In [ ]:
import json
import re

with open("original_posts.json", "r", encoding="utf-8") as f:
    threads = json.load(f)


keywords = [
    "read", "reading", "reader",

    "book", "books",

    "story", "stories",

    "fiction",

    "novel", "novels"
]

filtered_threads = []


for thread in threads:

    post = thread["post"]

    title = post.get("title", "").lower()

    words = set(re.findall(r"\b[a-z]+\b", title))

    if any(keyword in words for keyword in keywords):

        filtered_threads.append(thread)

print("Original posts:", len(threads))
print("Filtered posts:", len(filtered_threads))

with open("filtered_posts.json", "w", encoding="utf-8") as f:
    json.dump(
        filtered_threads,
        f,
        ensure_ascii=False,
        indent=4
    )

reading/read/reader filtering:

In [ ]:
import json
import re
with open("filtered_posts.json", "r", encoding="utf-8") as f:
    threads = json.load(f)


keywords = [
    "read", "reading", "reader"

]

filtered_threads = []


for thread in threads:

    post = thread["post"]

    title = post.get("title", "").lower()

    words = set(re.findall(r"\b[a-z]+\b", title))

    if any(keyword in words for keyword in keywords):

        filtered_threads.append(thread)


print("Original posts:", len(threads))
print("Filtered posts:", len(filtered_threads))

with open("filtered_posts(reading).json", "w", encoding="utf-8") as f:
    json.dump(
        filtered_threads,
        f,
        ensure_ascii=False,
        indent=4
    )

the other keywords filtering:

**book/ story/ fiction/ novel**

In [ ]:
import json
import re

with open("filtered_posts.json", "r", encoding="utf-8") as f:
    threads = json.load(f)

with open("filtered_posts(reading).json", "r", encoding="utf-8") as f:
    reading_threads = json.load(f)


reading_ids = {
    thread["post"]["id"]
    for thread in reading_threads
}

keywords = [
    "book", "books",
    "story", "stories",
    "fiction",
    "novel", "novels"
]

filtered_threads = []

for thread in threads:

    post = thread["post"]

    if post.get("id") in reading_ids:
        continue

    title = post.get("title", "").lower()

    words = set(re.findall(r"\b[a-z]+\b", title))

    if any(keyword in words for keyword in keywords):
        filtered_threads.append(thread)


print("Original filtered posts:", len(threads))
print("Reading posts:", len(reading_threads))
print("Other posts:", len(filtered_threads))

with open("filtered_posts(the other).json", "w", encoding="utf-8") as f:
    json.dump(
        filtered_threads,
        f,
        ensure_ascii=False,
        indent=4
    )

## **2.2 collecting comments of each filtered post**

key point:

the posts in "**filtered dataset.docx**" are manually selected from "**filtered_posts(reading).json**".

researchers should prepare **filtered dataset.docx** before running this section.

In [ ]:
!pip install python-docx

In [ ]:
from docx import Document
import requests
import json
import re
import time


headers = {
    "User-Agent": "Research_Project"
}


COMMENT_API = "https://arctic-shift.photon-reddit.com/api/comments/search"


DOCX_FILE = "filtered dataset.docx"
POST_FILE = "original_posts.json"

OUTPUT_FILE = "post&comments dataset.json"


doc = Document(DOCX_FILE)

text = "\n".join(
    p.text for p in doc.paragraphs
)

selected_ids = re.findall(
    r"Post ID:\s*([a-zA-Z0-9]+)",
    text
)

selected_ids = list(dict.fromkeys(selected_ids))


print("Selected posts:", len(selected_ids))


with open(
    POST_FILE,
    "r",
    encoding="utf-8"
) as f:
    original_posts = json.load(f)



post_dict = {}

for item in original_posts:

    post = item["post"]

    post_dict[post["id"]] = post



print(
    "Original posts loaded:",
    len(post_dict)
)


def fetch_comments(post_id):

    params = {
        "link_id": post_id,
        "limit": "auto",
        "sort": "asc"
    }


    response = requests.get(
        COMMENT_API,
        params=params,
        headers=headers
    )


    if response.status_code != 200:

        print(
            "Comment failed:",
            post_id,
            response.status_code
        )

        return []


    return response.json().get(
        "data",
        []
    )


results = []


for i, post_id in enumerate(
    selected_ids,
    start=1
):

    print(
        f"\n[{i}/{len(selected_ids)}]",
        post_id
    )

    post = post_dict.get(post_id)


    if post is None:

        print(
            "Post not found in original file"
        )

        continue



    comments = fetch_comments(post_id)


    print(
        "Comments:",
        len(comments)
    )


    results.append(
        {
            "post": post,
            "comments": comments
        }
    )


    time.sleep(0.5)



with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    "Saved posts:",
    len(results)
)

dataset cleaning:

In [ ]:
import json


input_file = "post&comments dataset.json"
output_file = "cleaned dataset.json"


with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)


clean_data = []


for item in data:

    post = item["post"]

    clean_post = {
        "id": post.get("id"),
        "title": post.get("title"),
        "selftext": post.get("selftext")
    }


    clean_comments = []

    for c in item["comments"]:

        clean_comments.append(
            {
                "id": c.get("id"),
                "link_id": c.get("link_id"),
                "parent_id": c.get("parent_id"),
                "body": c.get("body")
            }
        )


    clean_data.append(
        {
            "post": clean_post,
            "comments": clean_comments
        }
    )


with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        clean_data,
        f,
        ensure_ascii=False,
        indent=4
    )


print("ok")

docx trasforming:

In [ ]:
from docx import Document
import json


input_file = "cleaned dataset.json"
output_file = "cleaned dataset.docx"


with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)


doc = Document()


for i, item in enumerate(data, start=1):

    post = item["post"]
    comments = item["comments"]


    doc.add_heading(
        f"Post {i}",
        level=1
    )


    doc.add_paragraph(
        f"Post ID: {post.get('id')}"
    )


    doc.add_paragraph(
        "Title:"
    )

    doc.add_paragraph(
        post.get("title", "")
    )


    doc.add_paragraph(
        "Content:"
    )

    doc.add_paragraph(
        post.get("selftext", "")
    )



    doc.add_heading(
        "Comments",
        level=2
    )


    if comments:

        for j, comment in enumerate(comments, start=1):

            doc.add_paragraph(
                f"Comment {j}"
            )

            doc.add_paragraph(
                f"Comment ID: {comment.get('id')}"
            )

            doc.add_paragraph(
                comment.get("body", "")
            )

            doc.add_paragraph(
                "--------------------"
            )

    else:

        doc.add_paragraph(
            "No comments"
        )


    doc.add_paragraph(
        "--------------------------------"
    )



doc.save(output_file)


print("complete:", output_file)